# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
%load_ext autoreload
%autoreload 2
import molpot as mpot

/opt/conda/lib/python3.11/site-packages/torchdata/datapipes/__init__.py:18: UserWarning: 
################################################################################
WARNING!
The 'datapipes', 'dataloader2' modules are deprecated and will be removed in a
future torchdata release! Please see https://github.com/pytorch/data/issues/1196
to learn more and leave feedback.
################################################################################

  deprecation_warning()


In [2]:
# 1. get QM9 dataset
qm9_ds = mpot.dataset.QM9(
    save_dir="data/qm9",
    total=9,
    device="cpu"
)
# TODO: super ulgy here, change after torchdata settle down
dp = qm9_ds.get_pipeline()
max_cutoff = 5.0
dp = dp.calc_nblist(max_cutoff).calc_dist()
dp = dp.batch(3)
dp = dp.collate_frame()
train_dp, valid_dp = dp.random_split({"train": 0.8, "valid": 0.2}, seed=0)
# 2. wrap with DataLoader

train_dl = mpot.DataLoader(train_dp)
valid_dl = mpot.DataLoader(valid_dp)

In [3]:
data_inspector = mpot.inspector.DataInspector(qm9_ds)
data_inspector.summary()

number of data: 9

                           dataset: qm9                            
┏━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ label ┃      type       ┃   unit    ┃          comment          ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   A   │ <class 'float'> │    GHz    │   rotational_constant_A   │
│   B   │ <class 'float'> │    GHz    │   rotational_constant_B   │
│   C   │ <class 'float'> │    GHz    │   rotational_constant_C   │
│  mu   │ <class 'float'> │   Debye   │       dipole_moment       │
│ alpha │ <class 'float'> │    a0     │ isotropic_polarizability  │
│ homo  │ <class 'float'> │  hartree  │           homo            │
│ lumo  │ <class 'float'> │  hartree  │           lump            │
│  gap  │ <class 'float'> │  hartree  │            gap            │
│  r2   │ <class 'float'> │    a0     │ electronic_spatial_extent │
│ zpve  │ <class 'float'> │  hartree  │           zpve            │
│  U0   │ <class 'float'> │  hartree  │         energy_U0         │
│   U   │ <class 'float'> │  hartree  │         energy_U          │
│   H   │ <class 'float'> │  hartree  │        enthalpy_H         │
│   G   │ <class 'float'> │  hartree  │        free_energy        │
│  Cv   │ <class 'float'> │ cal/mol/K │       heat_capacity       │
└───────┴─────────────────┴───────────┴───────────────────────────┘

In [4]:
for data in dp:
    print(data['atoms'])
    print(data['bonds'])
    print(data['pairs'])
    print(data['box'])
    print(data['targets'])
    break

TensorDict(
    fields={
        Z: Tensor(shape=torch.Size([12]), device=cpu, dtype=torch.int64, is_shared=False),
        batch_mask: Tensor(shape=torch.Size([12]), device=cpu, dtype=torch.int64, is_shared=False),
        idx: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.int64, is_shared=False),
        n_atoms: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.int64, is_shared=False),
        xyz: Tensor(shape=torch.Size([12, 3]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)
TensorDict(
    fields={
    },
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)
TensorDict(
    fields={
        diff: Tensor(shape=torch.Size([38, 3]), device=cpu, dtype=torch.float32, is_shared=False),
        dist: Tensor(shape=torch.Size([38]), device=cpu, dtype=torch.float32, is_shared=False),
        i: Tensor(shape=torch.Size([38]), device=cpu, dtype=torch.int64, is_shared=False),
        j: Tenso

In [5]:
data['atoms']

TensorDict(
    fields={
        Z: Tensor(shape=torch.Size([12]), device=cpu, dtype=torch.int64, is_shared=False),
        batch_mask: Tensor(shape=torch.Size([12]), device=cpu, dtype=torch.int64, is_shared=False),
        idx: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.int64, is_shared=False),
        n_atoms: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.int64, is_shared=False),
        xyz: Tensor(shape=torch.Size([12, 3]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)

In [6]:
for batch in train_dl:
    print(batch.keys())
    break

_StringKeys(dict_keys(['atoms', 'bonds', 'angles', 'dihedrals', 'impropers', 'pairs', 'box', 'targets', 'predicts']))


## Setting up the model

In [7]:
cutoff = 5.0
n_rbf = 10
arch = mpot.nnp.PiNet(
    depth=4,
    basis_fn=mpot.nnp.GaussianRBF(n_rbf, cutoff),
    cutoff_fn=mpot.nnp.CosineCutoff(cutoff),
    pp_nodes=[16],
    pi_nodes=[16],
    ii_nodes=[16],
    max_atomtypes=10,
)
energy_readout = mpot.nnp.Atomwise(16, 1, [], from_key=("pinet", "p1"), to_key=("predicts", "U0"))

model = mpot.PotentialSeq("pinet", arch, energy_readout)

In [12]:
import torch

loss_fn = mpot.engine.loss.loss_wrapper(input_=("predicts", "U0"), target=("targets", "U0"))(torch.nn.MSELoss())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, 0.994)

trainer = mpot.PotentialTrainer(
    "pinet-qm9",
    model,
    loss_fn,
    optimizer,
    scheduler,
    root_dir="runs",
)
trainer.fix.register(
    mpot.engine.fix.TensorBoardFix(
        every_n_steps=100,
        log_dir="runs/pinet-qm9",
        outputs=["epoch_speed", "e_mae", "loss", "valid_loss"],
    ),
    trainer.Stage.after_iter,
)
trainer.fix.register(
    mpot.engine.fix.EpochSpeed(every_n_epochs=1), trainer.Stage.after_epoch
)
trainer.fix.register(
    mpot.engine.fix.StepSpeed(every_n_steps=1), trainer.Stage.after_epoch
)
trainer.fix.register(
    mpot.engine.fix.MAE(
        output_key="e_mae", every_n_steps=100, result_key=("predicts", "U0"), target_key=("targets", "U0")
    ),
    trainer.Stage.after_iter
)
# TODO: ckpt, valid
trainer.fix.register(
    mpot.engine.fix.Validation(1, valid_dl), trainer.Stage.after_epoch
)

In [16]:
%load_ext tensorboard
inputs = trainer.train(train_dl, 1000)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


# Training a Classic Potential

This section introduces how to use `MolPot` to train a classic forcefield

In [ ]:
import molpy as mp

In [ ]:
ff = mp.ForceField("gaff")

atomstyle = ff.def_atomstyle("full")
C = atomstyle.def_atomtype("C", 1, {"mass": 12.01})
H = atomstyle.def_atomtype("H", 2, {"mass": 1.01})
O = atomstyle.def_atomtype("O", 3, {"mass": 16.00})
N = atomstyle.def_atomtype("N", 4, {"mass": 14.01})

bondstyle = ff.def_bondstyle("harmonic")